In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import pytz

def show_dashboard_chart(csv_file_path):
    # 1. Check Current Time in IST
    ist_tz = pytz.timezone('Asia/Kolkata')
    current_time_ist = datetime.now(ist_tz)
    
    # 2. Time-Gate Logic: Only execute between 15:00 (3 PM) and 17:00 (5 PM)
    if not (15 <= current_time_ist.hour < 17):
        print(f"Graph is currently hidden. It is only available between 3 PM and 5 PM IST. (Current time: {current_time_ist.strftime('%I:%M %p')} IST)")
        return  # Exit the function, preventing the graph from rendering in the dashboard

    # 3. Load and Clean Data
    df = pd.read_csv(csv_file_path)

    # Clean 'Size' (Convert to Megabytes)
    def clean_size(size):
        if isinstance(size, str):
            if 'M' in size:
                return float(size.replace('M', ''))
            elif 'k' in size:
                return float(size.replace('k', '')) / 1024 # Convert kb to MB
        return np.nan
    df['Size_MB'] = df['Size'].apply(clean_size)

    # Clean 'Last Updated' (Convert to datetime)
    df['Last Updated'] = pd.to_datetime(df['Last Updated'], errors='coerce')

    # Clean 'Installs' and 'Reviews' (Convert to numeric)
    df['Installs'] = df['Installs'].astype(str).str.replace('+', '').str.replace(',', '')
    df['Installs'] = pd.to_numeric(df['Installs'], errors='coerce')
    df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce')

    # 4. Apply Initial Filters: Size >= 10M and Last Updated in January
    filtered_df = df[(df['Size_MB'] >= 10) & (df['Last Updated'].dt.month == 1)].copy()

    # 5. Group by Category
    category_stats = filtered_df.groupby('Category').agg({
        'Installs': 'sum',
        'Rating': 'mean',
        'Reviews': 'sum'
    }).reset_index()

    # 6. Apply Rating Filter: Average Rating >= 4.0
    category_stats = category_stats[category_stats['Rating'] >= 4.0]

    # 7. Get Top 10 Categories by Total Installs
    top_10_cats = category_stats.nlargest(10, 'Installs')

    # 8. Render the Grouped Bar Chart (Dual Y-Axis)
    fig, ax1 = plt.subplots(figsize=(12, 6))
    ax2 = ax1.twinx() # Create a secondary y-axis to handle the review counts scale

    x = np.arange(len(top_10_cats))
    width = 0.35

    bar1 = ax1.bar(x - width/2, top_10_cats['Rating'], width, color='skyblue', label='Average Rating')
    bar2 = ax2.bar(x + width/2, top_10_cats['Reviews'], width, color='lightgreen', label='Total Reviews')

    # Formatting axes and labels
    ax1.set_xticks(x)
    ax1.set_xticklabels(top_10_cats['Category'], rotation=45, ha='right')
    ax1.set_ylabel('Average Rating')
    ax2.set_ylabel('Total Reviews Count')
    ax1.set_title('Top 10 App Categories: Avg Rating vs Total Reviews (Jan Updates, Size $\ge$ 10M, Rating $\ge$ 4.0)')

    # Add a combined legend
    lines_labels = [ax.get_legend_handles_labels() for ax in [ax1, ax2]]
    lines, labels = [sum(lol, []) for lol in zip(*lines_labels)]
    ax1.legend(lines, labels, loc='upper left')

    plt.tight_layout()
    plt.show() # In a dashboard like Streamlit, replace plt.show() with st.pyplot(fig)

# Usage
show_dashboard_chart('googleplaystore.csv')

<>:69: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<>:69: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
C:\Users\jagdi\AppData\Local\Temp\ipykernel_7448\3340611351.py:69: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
  ax1.set_title('Top 10 App Categories: Avg Rating vs Total Reviews (Jan Updates, Size $\ge$ 10M, Rating $\ge$ 4.0)')


Graph is currently hidden. It is only available between 3 PM and 5 PM IST. (Current time: 01:40 PM IST)


In [2]:
show_dashboard_chart('googleplaystore.csv')

Graph is currently hidden. It is only available between 3 PM and 5 PM IST. (Current time: 01:41 PM IST)
